# 14 · SWE-bench-style patch evaluation

The SWE-bench loop — a model produces a **unified diff**, you
`git apply` it to a repo and re-run the test suite, scoring **FAIL → PASS**.

**Key concepts**
- **Patch as the artifact** — SWE-bench's prediction is a *diff*, applied with
  `git apply`, not a rewritten file. This is the faithful, scalable representation
  for real multi-file bug fixes.
- **Repository-level reward** — the reward is the repo's own test suite going from
  failing to passing, in a realistic environment.
- **Why microvm** — real repos need `git`, `pip`, and system tools → a full-OS VM.
  (For real instances you'd `git clone <repo>@<commit>` instead of the inline repo.)

**How it works:** lay down a buggy repo → run tests (FAIL) → model returns a diff →
`git apply` → re-run tests (PASS?).

**Prerequisites:** the `microvm` backend — `RAITBOX_MICROVM_IMAGE` +
`RAITBOX_EXECUTION_ROLE` (internet egress). See `docs/INTEGRATION.md`.

In [ ]:
%pip install -q raitbox

### Configuration & fixture
Bedrock client and a tiny buggy repo (`calc.py`) plus its unittest suite.

In [ ]:
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]
EXEC_ROLE = os.environ["RAITBOX_EXECUTION_ROLE"]
INTERNET_EGRESS = os.environ.get(
    "RAITBOX_EGRESS",
    f"arn:aws:lambda:{REGION}:aws:network-connector:aws-network-connector:INTERNET_EGRESS",
)
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

# A tiny repo with a bug: sum_even filters odds (n % 2 == 1). Tests expect even-sum.
BUGGY_SRC = "def sum_even(nums):\n    return sum(n for n in nums if n % 2 == 1)\n"
TEST_SRC = "\n".join([
    "import unittest",
    "from calc import sum_even",
    "class T(unittest.TestCase):",
    "    def test_basic(self):",
    "        self.assertEqual(sum_even([1,2,3,4]), 6)",
    "    def test_empty(self):",
    "        self.assertEqual(sum_even([]), 0)",
    "if __name__ == '__main__':",
    "    unittest.main()",
])

### Set up the repo & see it fail
Launch a MicroVM, install `git`, commit the buggy repo, and run the suite — it fails.

In [ ]:
sbx = Sandbox.create(backend="microvm", template=IMAGE_ARN, execution_role_arn=EXEC_ROLE,
                     egress_connectors=[INTERNET_EGRESS], region_name=REGION, timeout=1800)
print("sandbox", sbx.sandbox_id)

print(sbx.commands.run("dnf install -y git >/dev/null 2>&1; git --version").stdout)
sbx.files.write("/repo/calc.py", BUGGY_SRC)
sbx.files.write("/repo/test_calc.py", TEST_SRC)
sbx.commands.run("cd /repo && git init -q && git add -A && "
                 "git -c user.email=a@b.c -c user.name=x commit -qm init")

before = sbx.commands.run("cd /repo && python -m unittest -q 2>&1 || true")
print("BEFORE:\n", before.stdout[-500:])

### Get a patch & apply it
Ask the model for a unified diff, then `git apply` it to the repo.

In [ ]:
def ask_patch(failing_output: str, source: str) -> str:
    system = [{"text": "You are fixing a bug. Reply with ONLY a unified diff (git diff "
               "format, paths a/calc.py b/calc.py) that fixes the failing tests. No prose, no fences."}]
    user = (f"Failing test output:\n{failing_output}\n\nFile calc.py:\n{source}\n\n"
            "Return the unified diff that makes the tests pass.")
    resp = bedrock.converse(modelId=MODEL_ID, system=system,
                            messages=[{"role": "user", "content": [{"text": user}]}],
                            inferenceConfig={"maxTokens": 800, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    m = re.search(r"(diff --git .*|--- a/.*)", text, re.DOTALL)
    return (m.group(0) if m else text).strip() + "\n"


patch = ask_patch(before.stdout, BUGGY_SRC)
print("PATCH:\n", patch)
sbx.files.write("/repo/fix.patch", patch)
applied = sbx.commands.run("cd /repo && git apply fix.patch 2>&1 && echo APPLIED || echo APPLY_FAILED")
print(applied.stdout)

## Re-run the suite — did the patch resolve it?

In [ ]:
after = sbx.commands.run("cd /repo && python -m unittest -q 2>&1 || true")
print("AFTER:\n", after.stdout[-500:])
resolved = after.exit_code == 0 and "OK" in after.stdout
print("\nRESULT:", "FAIL -> PASS (resolved)" if resolved else "still failing")
sbx.kill()

## Recap & where to go next
You ran a patch-based, test-suite-graded coding eval end to end.

**Related:** `06` (iterate to green) · `15` (the same loop as a reusable RL env) ·
`09` (compare this execution reward to a similarity reward).